# Controlled Language Probabilities Across Layer Windows

LangDist-only story plots for synthetic controlled LangDist. This notebook loads existing `lang_probs_<revision>.pt` artifacts; it does not run model inference and it does not expand the target-string menu artifacts.

Copy, Cloze, and Translation are cached and plotted separately. Translation is intentionally last because it is the largest cache.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import (
    MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER,
    diagnose_story_distribution_sums,
    discover_eval_runs,
    load_lang_probs_artifact,
    micro_average_langdist_categories,
    micro_average_story_row_values,
    select_latest_unique_runs,
    synthetic_model_display_name,
    translation_target_from_data_source,
)
from vis import (
    overlay_story_category_max_markers,
    plot_story_category_bar_grid,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

LOG_ROOT = REPO_ROOT / "logs" / "evals"
CACHE_DIR = REPO_ROOT / ".analysis_cache" / "controlled_language_probs_layer_windows"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_APPENDIX_FIG_DIR = REPO_ROOT / "figs" / "paper" / "appendix"

CONTROLLED_LANGDIST_SCORING_MODE = "start_tokens_only"
CONTROLLED_LANGDIST_PAPER_OUTPUTS = {
    "copy": "06_copy_layer_windows",
    "cloze": "07_cloze_layer_windows",
    "translation": "08_translation_layer_windows",
}
CONTROLLED_LANGDIST_SYNTHETIC_ROOT = "data/wendler_2024_data/common69"
CONTROLLED_LANGDIST_EXPECTED_SYNTHETIC_LANGS = [
    "ar", "bg", "cs", "de", "en", "es", "fa", "fi", "fr", "gl", "hi", "id", "is",
    "it", "ja", "ko", "mr", "pl", "pt_br", "ru", "sr", "sv", "th", "tr", "uk", "ur", "zh",
]
CONTROLLED_LANGDIST_REQUIRE_FULL_SYNTHETIC_RUN = True

COPY_CLOZE_DATA_SOURCES = ["copy", "cloze"]
TRANSLATION_TARGET_LANGS = ["ar", "en", "fr", "hi", "ru", "tr", "zh"]
TRANSLATION_DATA_SOURCES = [f"translation_to_{lang}" for lang in TRANSLATION_TARGET_LANGS]
CONTROLLED_LANGDIST_DATA_SOURCES = [*COPY_CLOZE_DATA_SOURCES, *TRANSLATION_DATA_SOURCES]

# Match Fig. 4's low-to-high INCLUDE ranking for the available synthetic models.
CONTROLLED_LANGDIST_MODEL_ORDER = [
    model for model in MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER
    if model not in {"GPT-2", "GPT-2 XL", "OLMo-2-1124-7B"}
]
TASK_ORDER = ["copy", "cloze", "translation"]
TASK_LABELS = {"copy": "Copy", "cloze": "Cloze", "translation": "Translation"}
BAR_ORDER = ["task_relevant", "english", "other"]
BAR_LABELS = {
    "task_relevant": "Avg. P(L=task-relevant)",
    "english": "P(L=English)",
    "other": "Avg. P(L=other)",
}
BAR_COLORS = {
    "task_relevant": "#2f7ed8",
    "english": "#d95f02",
    "other": "#6a6a6a",
}
CONTROLLED_LANGDIST_LAYER_WINDOWS = [
    {
        "slug": "first50",
        "label": "First 50% Layers",
        "layer_min": 0.00,
        "layer_max": 0.50,
        "include_min": True,
        "include_max": False,
    },
    {
        "slug": "middle50_75",
        "label": "50%-75% Layers",
        "layer_min": 0.50,
        "layer_max": 0.75,
        "include_min": True,
        "include_max": False,
    },
    {
        "slug": "final25",
        "label": "75%-100% Layers",
        "layer_min": 0.75,
        "layer_max": 1.00,
        "include_min": True,
        "include_max": True,
    },
]

TASK_CACHE_PATHS = {
    "copy": CACHE_DIR / "copy_langdist_category_by_layer.parquet",
    "cloze": CACHE_DIR / "cloze_langdist_category_by_layer.parquet",
    "translation": CACHE_DIR / "translation_langdist_category_by_layer.parquet",
}
USE_CACHE = True
FORCE_REBUILD = False
EXCLUDE_ENGLISH_PROMPTS_OR_TARGETS = True

set_matplotlib_paper_font()


## Helpers

These helpers select latest matching evals by config, then aggregate saved language distributions into three categories. For translation, `task_relevant` is the mean over the source prompt language and requested target language, excluding English-filtered cases.


In [ ]:
def _config_list_key(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    try:
        return ",".join(sorted(str(item) for item in value))
    except TypeError:
        return str(value)


def _is_uncapped(value):
    return value in (None, 0)


def _matches_controlled_langdist_config(config):
    if not CONTROLLED_LANGDIST_REQUIRE_FULL_SYNTHETIC_RUN:
        return isinstance(config, dict)
    if not isinstance(config, dict):
        return False
    if config.get("synthetic_root") != CONTROLLED_LANGDIST_SYNTHETIC_ROOT:
        return False
    if _config_list_key(config.get("synthetic_langs")) != _config_list_key(CONTROLLED_LANGDIST_EXPECTED_SYNTHETIC_LANGS):
        return False
    if not _is_uncapped(config.get("max_prompts")):
        return False
    if not _is_uncapped(config.get("max_prompts_per_lang")):
        return False
    return True


def _add_target_string_config_columns(manifest):
    out = manifest.copy()
    out["data_source"] = out["config"].map(lambda c: c.get("data_source") if isinstance(c, dict) else None)
    out["target_string_scoring_mode"] = out["config"].map(
        lambda c: c.get("target_string_scoring_mode", "multi_token_teacher_forced")
        if isinstance(c, dict)
        else None
    )
    out["decoding_lens"] = out["config"].map(lambda c: c.get("decoding_lens") if isinstance(c, dict) else None)
    out["synthetic_root"] = out["config"].map(lambda c: c.get("synthetic_root") if isinstance(c, dict) else None)
    out["synthetic_langs_key"] = out["config"].map(
        lambda c: _config_list_key(c.get("synthetic_langs")) if isinstance(c, dict) else ""
    )
    out["max_prompts"] = out["config"].map(lambda c: c.get("max_prompts") if isinstance(c, dict) else None)
    out["max_prompts_per_lang"] = out["config"].map(
        lambda c: c.get("max_prompts_per_lang") if isinstance(c, dict) else None
    )
    return out


def load_target_string_langdist_manifest(*, data_sources=CONTROLLED_LANGDIST_DATA_SOURCES, log_root=LOG_ROOT):
    """Select complete target-string LangDist runs for the story plots."""
    manifest = discover_eval_runs(log_root=log_root, require_prompt_metrics=True)
    if manifest.empty:
        return manifest

    manifest = _add_target_string_config_columns(manifest)
    manifest = manifest[
        manifest["config"].map(
            lambda c: isinstance(c, dict)
            and c.get("do_decoding") is True
            and c.get("decoding_mapping") == "target_string"
        )
    ].copy()
    manifest = manifest[manifest["config"].map(_matches_controlled_langdist_config)].copy()
    manifest = manifest[
        manifest["data_source"].isin(list(data_sources))
        & manifest["target_string_scoring_mode"].eq(CONTROLLED_LANGDIST_SCORING_MODE)
        & manifest["decoding_lens"].fillna("raw_logitlens").eq("raw_logitlens")
    ].copy()
    if manifest.empty:
        return manifest.reset_index(drop=True)

    manifest["display_model_name"] = manifest["model_name"].map(synthetic_model_display_name)
    manifest["translation_target_lang"] = manifest["data_source"].map(translation_target_from_data_source)
    manifest["task"] = np.where(manifest["data_source"].isin(COPY_CLOZE_DATA_SOURCES), manifest["data_source"], "translation")
    manifest = select_latest_unique_runs(
        manifest,
        extra_keys=[
            "data_source",
            "target_string_scoring_mode",
            "synthetic_root",
            "synthetic_langs_key",
            "max_prompts",
            "max_prompts_per_lang",
        ],
    )
    order = {name: idx for idx, name in enumerate(CONTROLLED_LANGDIST_MODEL_ORDER)}
    manifest["display_model_sort"] = manifest["display_model_name"].map(order).fillna(len(order)).astype(int)
    return manifest.sort_values(
        ["task", "data_source", "display_model_sort", "display_model_name", "run_mtime_ns", "exp_id"]
    ).reset_index(drop=True)


def _category_indices(langs, *, prompt_lang, requested_tgt_lang):
    lang_to_idx = {lang: idx for idx, lang in enumerate(langs)}
    relevant_langs = {prompt_lang, requested_tgt_lang}
    relevant_langs.discard(None)
    relevant_langs.discard("en")
    relevant_indices = [lang_to_idx[lang] for lang in sorted(relevant_langs) if lang in lang_to_idx]
    english_indices = [lang_to_idx["en"]] if "en" in lang_to_idx else []
    claimed = set(relevant_indices) | set(english_indices)
    other_indices = [idx for idx in range(len(langs)) if idx not in claimed]
    return {
        "task_relevant": relevant_indices,
        "english": english_indices,
        "other": other_indices,
    }


def build_target_string_langdist_category_rows(manifest, *, task):
    """Expand saved LangDists into prompt-layer category rows."""
    columns = [
        "task",
        "data_source",
        "display_model_name",
        "model_name",
        "exp_id",
        "prompt_id",
        "prompt_lang",
        "requested_tgt_lang",
        "layer",
        "layer_norm",
        "prob_category",
        "category_lang_count",
        "category_prob_sum",
        "category_prob_sq_sum",
        "prob_langdist",
        "prob_langdist_max",
    ]
    if manifest.empty:
        return pd.DataFrame(columns=columns)

    task_manifest = manifest[manifest["task"].eq(task)].copy()
    if task_manifest.empty:
        return pd.DataFrame(columns=columns)

    frames = []
    for run in tqdm(task_manifest.itertuples(index=False), total=len(task_manifest), desc=f"Loading {task} LangDist", unit="run"):
        artifact_path = Path(run.exp_path) / f"lang_probs_{run.revision}.pt"
        if not artifact_path.exists():
            print(f"[{task}] missing lang_probs artifact for {run.exp_id}: {artifact_path}", flush=True)
            continue
        artifact = load_lang_probs_artifact(artifact_path)
        payload = artifact.get("methods", {}).get("decoding", {})
        if "probs" not in payload:
            print(f"[{task}] no decoding probs in {run.exp_id}", flush=True)
            continue

        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(torch.float32).numpy()
        layers = np.asarray(artifact.get("layer_indices") or list(range(probs.shape[1])), dtype=int)
        max_layer = max(int(layers.max()) if layers.size else probs.shape[1] - 1, 1)
        prompt_ids = np.asarray(artifact.get("prompt_ids") or [None] * probs.shape[0], dtype=object)
        prompt_langs = np.asarray(artifact.get("prompt_langs") or [None] * probs.shape[0], dtype=object)

        if task == "translation":
            requested_tgt_lang = run.translation_target_lang
        else:
            requested_tgt_lang = None

        print(
            f"[{task}] aggregating {run.exp_id}: {probs.shape[0]:,} prompts x {probs.shape[1]:,} layers x {probs.shape[2]:,} languages",
            flush=True,
        )

        prompt_lang_values = sorted({lang for lang in prompt_langs.tolist() if isinstance(lang, str)})
        for prompt_lang in tqdm(prompt_lang_values, desc=f"Aggregating {run.exp_id}", unit="lang", leave=False):
            effective_tgt_lang = requested_tgt_lang or prompt_lang
            if EXCLUDE_ENGLISH_PROMPTS_OR_TARGETS and (prompt_lang == "en" or effective_tgt_lang == "en"):
                continue
            prompt_indices = np.flatnonzero(prompt_langs == prompt_lang)
            if prompt_indices.size == 0:
                continue

            category_indices = _category_indices(langs, prompt_lang=prompt_lang, requested_tgt_lang=effective_tgt_lang)
            prompt_probs = probs[prompt_indices]
            base = pd.DataFrame(
                {
                    "prompt_id": np.repeat(prompt_ids[prompt_indices], len(layers)),
                    "prompt_lang": prompt_lang,
                    "requested_tgt_lang": effective_tgt_lang,
                    "layer": np.tile(layers, prompt_indices.size),
                    "layer_norm": np.tile(layers / float(max_layer), prompt_indices.size),
                }
            )

            for category, indices in category_indices.items():
                lang_count = len(indices)
                if lang_count:
                    category_probs = prompt_probs[:, :, indices]
                    category_sum = category_probs.sum(axis=-1)
                    category_sq_sum = np.square(category_probs).sum(axis=-1)
                    values = category_sum / float(lang_count)
                    max_values = category_probs.max(axis=-1)
                else:
                    category_sum = np.full((prompt_indices.size, len(layers)), np.nan, dtype=np.float32)
                    category_sq_sum = category_sum
                    values = category_sum
                    max_values = category_sum
                frame = base.copy()
                frame["task"] = task
                frame["data_source"] = run.data_source
                frame["display_model_name"] = run.display_model_name
                frame["model_name"] = run.model_name
                frame["exp_id"] = run.exp_id
                frame["prob_category"] = category
                frame["category_lang_count"] = lang_count
                frame["category_prob_sum"] = category_sum.reshape(-1)
                frame["category_prob_sq_sum"] = category_sq_sum.reshape(-1)
                frame["prob_langdist"] = values.reshape(-1)
                frame["prob_langdist_max"] = max_values.reshape(-1)
                frames.append(frame[columns])

    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True)


def load_or_build_target_string_category_cache(manifest, *, task):
    """Load a complete category cache or rebuild it from saved LangDists."""
    cache_path = TASK_CACHE_PATHS[task]
    if USE_CACHE and cache_path.exists() and not FORCE_REBUILD:
        cached = pd.read_parquet(cache_path)
        required_cache_columns = {"prob_langdist_max", "category_prob_sq_sum"}
        if required_cache_columns.issubset(cached.columns):
            print(f"Loading cached {task} LangDist category/layer table: {cache_path.relative_to(REPO_ROOT)}", flush=True)
            return cached
        print(f"Rebuilding {task} cache without exact micro-averaging statistics: {cache_path.relative_to(REPO_ROOT)}", flush=True)
    task_df = build_target_string_langdist_category_rows(manifest, task=task)
    task_df.to_parquet(cache_path, index=False)
    task_df.to_csv(cache_path.with_suffix(".csv"), index=False)
    print(f"Wrote {task} cache: {cache_path.relative_to(REPO_ROOT)}", flush=True)
    return task_df


def summarize_story_layer_windows(task_df, *, use_category_max=False):
    """Micro-average LangDist categories over each configured layer window."""
    summaries = []
    for window in CONTROLLED_LANGDIST_LAYER_WINDOWS:
        print(f"Building {window['label']} summary...", flush=True)
        common_kwargs = dict(
            group_cols=("task", "display_model_name"),
            layer_min=window["layer_min"],
            layer_max=window["layer_max"],
            include_min=window["include_min"],
            include_max=window["include_max"],
            keep_category_lang_count=True,
        )
        if use_category_max:
            summary = micro_average_story_row_values(
                task_df, prob_col="prob_langdist_max", **common_kwargs
            )
        else:
            summary = micro_average_langdist_categories(task_df, **common_kwargs)
        summary["layer_window"] = window["label"]
        summary["layer_window_slug"] = window["slug"]
        summaries.append(summary)
    if not summaries:
        return pd.DataFrame()
    return pd.concat(summaries, ignore_index=True)


def plot_target_string_story_grid(summary, max_summary, *, task):
    """Build and save one target-string story grid."""
    task_label = TASK_LABELS.get(task, task.title())
    exclusion_text = (
        "Cases with English as a source or target language are excluded."
        if task == "translation"
        else "English prompt cases are excluded."
    )
    fig = plot_story_category_bar_grid(
        summary,
        row_col="task",
        row_order=[task],
        col_col="layer_window",
        col_order=[window["label"] for window in CONTROLLED_LANGDIST_LAYER_WINDOWS],
        models=CONTROLLED_LANGDIST_MODEL_ORDER,
        category_order=BAR_ORDER,
        category_labels=BAR_LABELS,
        category_colors=BAR_COLORS,
        probability_label="Avg. Lang Prob across Prompts and Layers",
        shared_ylabel="Avg. Lang Prob across Prompts and Layers",
        shared_ylabel_x=0.055,
        title=f"[{task_label} task] Language Probabilities Across Layer Windows",
        subtitle=(
            f"{exclusion_text} Bars pool prompt-layer-language probabilities within each category;\n"
            "diamonds average prompt-layer category maxima; error bars show SE for both."
        ),
        unavailable_label=f"{task_label} LangDist",
        figsize=(14.5, 4.8),
        show_row_label_in_ylabel=False,
        legend_y=-0.010,
        bottom=0.36,
        top=0.76,
        title_y=0.995,
        subtitle_y=0.940,
        subtitle_linespacing=0.9,
        ylabel_pad=14.0,
    )
    if fig is not None:
        plot_models = [model for model in CONTROLLED_LANGDIST_MODEL_ORDER if model in set(summary["display_model_name"])]
        overlay_story_category_max_markers(
            fig,
            summary,
            max_summary,
            row_col="task",
            col_col="layer_window",
            row_order=[task],
            col_order=[window["label"] for window in CONTROLLED_LANGDIST_LAYER_WINDOWS],
            models=plot_models,
            category_order=BAR_ORDER,
            category_colors=BAR_COLORS,
            legend_y=-0.010,
            legend_fontsize=18,
            x_edge_padding=0.46,
            x_tick_shift_points=14.0,
        )
        save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / CONTROLLED_LANGDIST_PAPER_OUTPUTS[task])
        plt.show()
    return fig


def inspect_target_string_category_cache(task_df, *, task):
    """Print coverage and probability-mass diagnostics for a task cache."""
    print(f"{task} category/layer rows: {len(task_df):,}")
    if task_df.empty:
        return
    display(
        task_df.groupby(["task", "display_model_name"], dropna=False)
        .agg(prompts=("prompt_id", "nunique"), layers=("layer", "nunique"))
        .reset_index()
        .sort_values(["task", "display_model_name"])
    )
    print("Distribution reconstruction check: category mass should be close to 1.0.")
    display(
        diagnose_story_distribution_sums(
            task_df,
            prob_col="prob_langdist",
            group_cols=("task", "display_model_name"),
        )
    )


## Check Available Result Files

This cell only inspects completed result directories and `lang_probs` artifacts.


In [ ]:
manifest = load_target_string_langdist_manifest()
print(f"Selected controlled LangDist LangDist runs: {len(manifest):,}")
if manifest.empty:
    print("No matching completed runs found under logs/evals.")
else:
    manifest = manifest.copy()
    manifest["lang_probs_artifact_path"] = manifest.apply(
        lambda row: Path(row["exp_path"]) / f"lang_probs_{row['revision']}.pt",
        axis=1,
    )
    manifest["lang_probs_artifact_exists"] = manifest["lang_probs_artifact_path"].map(lambda path: Path(path).exists())
    display_cols = [
        "task",
        "data_source",
        "display_model_name",
        "exp_id",
        "lang_probs_artifact_exists",
    ]
    display(
        manifest
        .sort_values(["task", "data_source", "display_model_sort", "display_model_name"])
        [display_cols]
    )
    print("Coverage by task/data source:")
    display(
        manifest.groupby(["display_model_name", "data_source"], dropna=False)
        .size()
        .rename("runs")
        .reset_index()
        .pivot_table(index="display_model_name", columns="data_source", values="runs", fill_value=0)
        .reindex([m for m in CONTROLLED_LANGDIST_MODEL_ORDER if m in set(manifest["display_model_name"])])
        .fillna(0)
        .astype(int)
    )


## Copy

Copy is cached and plotted first.


In [ ]:
copy_df = load_or_build_target_string_category_cache(manifest, task="copy")
inspect_target_string_category_cache(copy_df, task="copy")
copy_summary = summarize_story_layer_windows(copy_df)
copy_max_summary = summarize_story_layer_windows(copy_df, use_category_max=True)
print(f"copy summary rows: {len(copy_summary):,}")
_copy_fig = plot_target_string_story_grid(copy_summary, copy_max_summary, task="copy")


## Cloze

Cloze is cached and plotted separately from copy.


In [ ]:
cloze_df = load_or_build_target_string_category_cache(manifest, task="cloze")
inspect_target_string_category_cache(cloze_df, task="cloze")
cloze_summary = summarize_story_layer_windows(cloze_df)
cloze_max_summary = summarize_story_layer_windows(cloze_df, use_category_max=True)
print(f"cloze summary rows: {len(cloze_summary):,}")
_cloze_fig = plot_target_string_story_grid(cloze_summary, cloze_max_summary, task="cloze")


## Translation

Translation is cached and plotted last. All configured `translation_to_*` runs are combined, excluding English source or requested-target cases.


In [ ]:
translation_df = load_or_build_target_string_category_cache(manifest, task="translation")
inspect_target_string_category_cache(translation_df, task="translation")
translation_summary = summarize_story_layer_windows(translation_df)
translation_max_summary = summarize_story_layer_windows(translation_df, use_category_max=True)
print(f"translation summary rows: {len(translation_summary):,}")
_translation_fig = plot_target_string_story_grid(translation_summary, translation_max_summary, task="translation")
